In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error

RANDOM_STATE = 42
TARGET = "ClosePrice"

## 0. Load Cleaned CRMLS and California School District Datasets

In [2]:
training_set = pd.read_parquet("../data/train_preprocessed.parquet")
testing_set = pd.read_parquet("../data/test_preprocessed.parquet")
train_df, test_df = training_set.copy(), testing_set.copy()
df = pd.read_parquet("../data/full_data_preprocessed.parquet")
original_models = pd.read_parquet("../data/baseline_model_performances.parquet")
print(f"Train dataset: {train_df.shape}, Test dataset: {test_df.shape}, Full Dataset: {df.shape} \n")

districts_raw = gpd.read_file("../data/California_School_District_Areas_2024-25.geojson")
print(f"Loaded {len(districts_raw)} districts from California School District Areas 2024-2025")
print(f"District Type Unique Entries: {districts_raw["DistrictType"].unique()}")

Train dataset: (81857, 30), Test dataset: (7442, 30), Full Dataset: (240901, 30) 

Loaded 937 districts from California School District Areas 2024-2025
District Type Unique Entries: ['Unified' 'Elementary' 'High']


In [3]:
districts = districts_raw[districts_raw["DistrictType"].isin(["Unified", "High"])].copy()
districts = districts[["DistrictType", "DistrictName", "geometry"]].to_crs("EPSG:4326")
print(f"Filtered to {len(districts)} Unified/High polygons")

Filtered to 421 Unified/High polygons


In [4]:
# raw unscaled lookup table
raw_lookup = df[["ListingKey", "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt", "Latitude", "Longitude"]]
raw_lookup = raw_lookup.rename(columns={
    "BedroomsTotal": "BedroomsTotal_raw",
    "BathroomsTotalInteger": "BathroomsTotalInteger_raw",
    "YearBuilt": "YearBuilt_raw",
    "Latitude": "Latitude_raw",
    "Longitude": "Longitude_raw",
})
n_before = len(raw_lookup)
raw_lookup = raw_lookup.drop_duplicates(subset="ListingKey", keep="first")
print(f"raw_lookup: {len(raw_lookup)} rows ({n_before - len(raw_lookup)} duplicate ListingKeys removed)")

raw_lookup: 240699 rows (202 duplicate ListingKeys removed)


#### Helper Functions:

In [5]:
def merge_raw_lookup(frame, lookup=raw_lookup):
    n_before = len(frame)
    merged = frame.merge(lookup, on="ListingKey", how="left")
    assert len(merged) == n_before, f"Merge changed row count: {n_before} -> {len(merged)} (duplicate key in lookup)"
    merged.index = frame.index
    return merged

# Build X and y train and test sets
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

def build_X_y(frame, NON_FEATURE_COLS, CATEGORICAL_COLS):
    encoder.fit(train_df[CATEGORICAL_COLS])
    numeric_part = frame.drop(columns=[c for c in [TARGET] + NON_FEATURE_COLS + CATEGORICAL_COLS if c in frame.columns])
    numeric_part = numeric_part.select_dtypes(include=[np.number])
    encoded = encoder.transform(frame[CATEGORICAL_COLS])
    X = hstack([csr_matrix(numeric_part.values), encoded]).tocsr()
    y = frame[TARGET].values
    feature_names = list(numeric_part.columns) + list(encoder.get_feature_names_out(CATEGORICAL_COLS))
    return X, y, feature_names

# Evaluate Model Performance Function
def evaluate_models(X_train,y_train,X_test,y_test, NEW_FEATURE):
    def evaluate_model(model, X_train, y_train, X_test, y_test, name):
        train_pred, test_pred = model.predict(X_train), model.predict(X_test)
        return {"model": name, "train_r2": r2_score(y_train, train_pred), "test_r2": r2_score(y_test, test_pred),
            "test_mae": mean_absolute_error(y_test, test_pred), "test_rmse": root_mean_squared_error(y_test, test_pred),
            "test_mape": mean_absolute_percentage_error(y_test, test_pred)}

    new_results = []
    linear_model = LinearRegression().fit(X_train, y_train)
    new_results.append(evaluate_model(linear_model, X_train, y_train, X_test, y_test, "linear regression"))

    tree_tuned = DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=RANDOM_STATE).fit(X_train, y_train)
    new_results.append(evaluate_model(tree_tuned, X_train, y_train, X_test, y_test, "decision tree"))

    rf_model = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE).fit(X_train, y_train)
    new_results.append(evaluate_model(rf_model, X_train, y_train, X_test, y_test, "random forest"))

    new_results_df = pd.DataFrame(new_results)
    new_results_df["New Features"] = [NEW_FEATURE for i in range(len(new_results_df))]

    return new_results_df.round(4)

## 1. Add Geographic School District Layer

In [6]:
def add_spatial_district(frame, districts):
    points_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(frame["Longitude_raw"], frame["Latitude_raw"]),
        crs="EPSG:4326",
        index=frame.index,
    )
    joined = gpd.sjoin(points_gdf, districts, how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]

    spatial_col = pd.Series(np.nan, index=frame.index, dtype=object)
    spatial_col.loc[joined.index] = joined["DistrictName"].values
    print(f"Real spatial match rate (before fallback): {spatial_col.notna().mean():.2%}")
    return spatial_col.fillna(frame["HighSchoolDistrict"])

train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)
train_df["SchoolDistrictSpatial"] = add_spatial_district(train_df,districts) # unedited Training and Testing Sets
test_df["SchoolDistrictSpatial"] = add_spatial_district(test_df, districts)

print(f"Train coverage: {train_df['SchoolDistrictSpatial'].notna().mean():.2%}")
print(f"Test coverage:  {test_df['SchoolDistrictSpatial'].notna().mean():.2%}")

Real spatial match rate (before fallback): 99.89%
Real spatial match rate (before fallback): 99.95%
Train coverage: 100.00%
Test coverage:  100.00%


In [7]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw"]
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "SchoolDistrictSpatial", "Levels"]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns]

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

# Create Comparison df to track model performances as features are added
og_models = original_models.copy()
og_models["New Features"] = ["Baseline" for i in range(len(og_models))]

comparison_df = pd.concat([og_models,evaluate_models(X_train, y_train, X_test, y_test, "SchoolDistrictSpatial")], ignore_index = True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial


In [8]:
agreement = (train_df["SchoolDistrictSpatial"] == train_df["HighSchoolDistrict"]).mean()
print(f"SchoolDistrictSpatial matches self-reported HighSchoolDistrict: {agreement:.1%}")

n_spatial = train_df["SchoolDistrictSpatial"].nunique()
n_self_reported = train_df["HighSchoolDistrict"].nunique()
print(f"Distinct spatial districts: {n_spatial}, distinct self-reported: {n_self_reported}")

mismatched = train_df[train_df["SchoolDistrictSpatial"] != train_df["HighSchoolDistrict"]]
mismatched[["HighSchoolDistrict", "SchoolDistrictSpatial"]].value_counts().head(20)

SchoolDistrictSpatial matches self-reported HighSchoolDistrict: 58.2%
Distinct spatial districts: 377, distinct self-reported: 405


HighSchoolDistrict        SchoolDistrictSpatial         
Unknown                   Desert Sands Unified              1652
William S. Hart Union     William S. Hart Union High        1496
Unknown                   San Diego Unified                 1428
Temecula Unified          Temecula Valley Unified           1187
Unknown                   Palm Springs Unified              1116
Antelope Valley Union     Antelope Valley Union High        1100
Menifee Union             Perris Union High                  871
Murrieta                  Murrieta Valley Unified            829
Newport Mesa Unified      Newport-Mesa Unified               787
Beaumont                  Beaumont Unified                   732
Unknown                   Los Angeles Unified                726
                          Grossmont Union High               705
Victor Valley Unified     Victor Valley Union High           663
Unknown                   Poway Unified                      488
                          Coachel

- Spatial feature is important when HighSchoolDistrict. is "Unknown"
- redundant when the difference is only in naming convention
- And very few convert High School and Elementary districts

In [9]:
def normalize_district_name(series):
    return (series.str.lower()
            .str.replace(r"\b(unified|union|high|school|district|joint)\b", "", regex=True)
            .str.replace(r"[^a-z]", "", regex=True))

has_self_reported = train_df["HighSchoolDistrict"].notna() & (train_df["HighSchoolDistrict"] != "Unknown")

filled_missing = (~has_self_reported).sum()
print(f"Rows where spatial join filled in a previously missing/Unknown value: {filled_missing} ({filled_missing/len(train_df):.1%})")

both_present = train_df[has_self_reported]
normalized_match_both_present = (
    normalize_district_name(both_present["HighSchoolDistrict"]) ==
    normalize_district_name(both_present["SchoolDistrictSpatial"])
)
print(f"True agreement rate, self-reported vs spatial, when both have an answer: {normalized_match_both_present.mean():.1%}")

Rows where spatial join filled in a previously missing/Unknown value: 11221 (13.7%)
True agreement rate, self-reported vs spatial, when both have an answer: 82.9%


In [10]:
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "SchoolDistrictSpatial", "Levels"]  # HighSchoolDistrict replaced, not added alongside
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns]

X_train, y_train, feature_names = build_X_y(train_df, NON_FEATURE_COLS, CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df, NON_FEATURE_COLS, CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df, evaluate_models(
    X_train, y_train, X_test, y_test, "SchoolDistrictSpatial (replacing HighSchoolDistrict)"
)], ignore_index=True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...


#### Imrpovement for the linear model with no meaningful cost elsewhere when joining HighSchoolDistrict (MLS origin) with SchoolDistrictSpatial (CA School District Data)
- Linear Regression test R2 went from 0.8420 -> 0.8442
- Decision Tree dipped slightly (0.7961 -> 0.7945, likely noise), 
- Random Forest is essentially flat with a small MAPE improvement. 

## 2. Create Bed to Bath Ratio Feature

In [11]:
# reset train_df and test_df after engineering the feature above
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

In [12]:
# Engineer Bed to Bath Ratio Feature
train_df["BedBathRatio"] = train_df["BedroomsTotal_raw"] / train_df["BathroomsTotalInteger_raw"].replace(0, np.nan)
test_df["BedBathRatio"] = test_df["BedroomsTotal_raw"] / test_df["BathroomsTotalInteger_raw"].replace(0, np.nan)

train_bed_bath_median = train_df["BedBathRatio"].median()
train_df["BedBathRatio"] = train_df["BedBathRatio"].fillna(train_bed_bath_median)
test_df["BedBathRatio"] = test_df["BedBathRatio"].fillna(train_bed_bath_median)

In [13]:
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels"] # Exclude Geographic layer engineered above
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 
# Tests only Bed Bath Ratio

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "BedBathRatio")], ignore_index = True )
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


#### Improvement for Linear Model, and Decision Tree and Random Forest Models stayed Roughly the same

## 3. Create Property Age feature

In [14]:
# reset train_df and test_df after engineering the features above
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

In [15]:
# Engineer Propert Age Feature
train_df["PropertyAgeAtSale"] = (train_df["CloseDate"].dt.year - train_df["YearBuilt_raw"]).clip(lower=0)
test_df["PropertyAgeAtSale"] = (test_df["CloseDate"].dt.year - test_df["YearBuilt_raw"]).clip(lower=0)

train_property_age_median = train_df["PropertyAgeAtSale"].median()
train_df["PropertyAgeAtSale"] = train_df["PropertyAgeAtSale"].fillna(train_property_age_median)
test_df["PropertyAgeAtSale"] = test_df["PropertyAgeAtSale"].fillna(train_property_age_median)

In [16]:
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels"] 
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 
# Tests only Property Age At Sale

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "PropertyAgeAtSale")], ignore_index = True )
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


#### Slight Improvement for Linear Regression and Decision Tree, negligble change in Random Forest

## 4. Create Amenity Count Feature

In [17]:
# reset train_df and test_df after engineering the features above
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

In [18]:
AMENITY_COLS = ["ViewYN", "PoolPrivateYN", "WaterfrontYN", "FireplaceYN", "NewConstructionYN"]
train_df["AmenityCount"] = train_df[AMENITY_COLS].sum(axis=1)
test_df["AmenityCount"] = test_df[AMENITY_COLS].sum(axis=1)

In [19]:
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels"] 
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 
# Tests only Property Age At Sale

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "AmenityCount")], ignore_index = True )
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


## 5. Building Coverage Ratio

In [20]:
# reset train_df and test_df after engineering the features above
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

In [21]:
train_df["BuildingCoverageRatio"] = train_df["LivingArea"] / train_df["LotSizeSquareFeet"].replace(0, np.nan)
test_df["BuildingCoverageRatio"] = test_df["LivingArea"] / test_df["LotSizeSquareFeet"].replace(0, np.nan)
train_med_cov = train_df["BuildingCoverageRatio"].median()
train_df["BuildingCoverageRatio"] = train_df["BuildingCoverageRatio"].fillna(train_med_cov)
test_df["BuildingCoverageRatio"] = test_df["BuildingCoverageRatio"].fillna(train_med_cov)

In [22]:
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels"] 
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 
# Tests only Property Age At Sale

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "BuildingCoverageRatio")], ignore_index = True )
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


## 6. K-Nearest Neighbors Comparable Price (CompPriceKNN)

In [23]:
# reset train_df and test_df after engineering the features above
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

COMP_K = 15     # nearest 15 neighbors by long and lat
coord_cols = ["Latitude_raw", "Longitude_raw"]
train_coords = train_df[coord_cols].values
train_prices = train_df["ClosePrice"].values

nn = NearestNeighbors(n_neighbors=COMP_K + 1, algorithm="ball_tree").fit(train_coords)
_, neighbor_idx = nn.kneighbors(train_coords)
neighbor_idx = neighbor_idx[:, 1:]  # drop self
train_df["CompPriceKNN"] = train_prices[neighbor_idx].mean(axis=1)

nn_test = NearestNeighbors(n_neighbors=COMP_K, algorithm="ball_tree").fit(train_coords)
_, test_neighbor_idx = nn_test.kneighbors(test_df[coord_cols].values)
test_df["CompPriceKNN"] = train_prices[test_neighbor_idx].mean(axis=1)

In [24]:
# CompPriceKNN would greatly improve DT and RF, but heavily reduce LR because 
# it had massive gaps in value between columns, unlike other previously normalized numeric features

comp_scaler = StandardScaler()
train_df["CompPriceKNN"] = comp_scaler.fit_transform(train_df[["CompPriceKNN"]])
test_df["CompPriceKNN"] = comp_scaler.transform(test_df[["CompPriceKNN"]])

In [25]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw", "Latitude_raw", "Longitude_raw"]
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels"] 
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 
# Tests only Property Age 

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "CompPriceKNN")], ignore_index = True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


#### Imrpovements for all models (greatest change in DT, then LR, then RF test R2)
R2 Change:
- LR: 0.8420 -> 0.8589
- DT: 0.7961 -> 0.8552
- RF: 0.8734 -> 0.8850

## 6.b CompPriceKNN per Square Feet

In [26]:
# Test CompPriceKNN per Lot Size (account for neighboring properties differing in size)

train_living_area = train_df["LivingArea"].values
comp_price_per_sqft = (train_prices / train_living_area)  # reuses train_prices/coords from CompPriceKNN
train_df["CompPricePerSqftKNN"] = comp_price_per_sqft[neighbor_idx[:, 1:]].mean(axis=1)
test_df["CompPricePerSqftKNN"] = comp_price_per_sqft[test_neighbor_idx].mean(axis=1)

In [27]:
train_df["CompPricePerSqftKNN"] = comp_scaler.fit_transform(train_df[["CompPricePerSqftKNN"]])
test_df["CompPricePerSqftKNN"] = comp_scaler.transform(test_df[["CompPricePerSqftKNN"]])

In [28]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw", "Latitude_raw", "Longitude_raw", "CompPriceKNN"] # Test without CompPriceKNN
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels"] 
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 
# Tests only Property Age 

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "CompPricePerSqftKNN")], ignore_index = True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


## 6.c Interaction Terms for Linear Model

In [29]:
train_df["LivingArea_x_CompPrice"] = train_df["LivingArea"] * train_df["CompPriceKNN"]
test_df["LivingArea_x_CompPrice"] = test_df["LivingArea"] * test_df["CompPriceKNN"]

train_df["YearBuilt_sq"] = train_df["YearBuilt_raw"] ** 2
test_df["YearBuilt_sq"] = test_df["YearBuilt_raw"] ** 2

In [33]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw", "Latitude_raw", "Longitude_raw", "CompPriceKNN", "CompPricePerSqftKNN"] # Test without CompPriceKNN or CPKNN/SqFt
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    train_pred, test_pred = model.predict(X_train), model.predict(X_test)
    return {"model": name, "train_r2": r2_score(y_train, train_pred), "test_r2": r2_score(y_test, test_pred),
            "test_mae": mean_absolute_error(y_test, test_pred), "test_rmse": root_mean_squared_error(y_test, test_pred),
            "test_mape": mean_absolute_percentage_error(y_test, test_pred)}

new_results = []
linear_model = LinearRegression().fit(X_train, y_train)
new_results.append(evaluate_model(linear_model, X_train, y_train, X_test, y_test, "linear regression"))
new_results_df = pd.DataFrame(new_results)
comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "LR Interaction Terms")], ignore_index = True)
comparison_df

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 7. Add MonthSin and MonthCos (Seasonality) and MonthsSinceStart (Linear Trend)

In [34]:
# Using Fourier Terms:
# Cyclical time encoding lets the model recognize December and January as adjacent rather than maximally distant (like 1 - 12 would be).

# reset train_df and test_df after engineering the features above
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

close_month_period = pd.PeriodIndex(train_df["CloseMonth"], freq="M")
test_month_period = pd.PeriodIndex(test_df["CloseMonth"], freq="M")

all_months_sorted = sorted(set(close_month_period) | set(test_month_period))
month_to_trend = {m: i for i, m in enumerate(all_months_sorted)}

train_df["MonthsSinceStart"] = close_month_period.map(month_to_trend)
test_df["MonthsSinceStart"] = test_month_period.map(month_to_trend)

train_df["MonthSin"] = np.sin(2 * np.pi * close_month_period.month / 12)
train_df["MonthCos"] = np.cos(2 * np.pi * close_month_period.month / 12)
test_df["MonthSin"] = np.sin(2 * np.pi * test_month_period.month / 12)
test_df["MonthCos"] = np.cos(2 * np.pi * test_month_period.month / 12)

In [35]:
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels"] 
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "Cyclical Time Encoding")], ignore_index = True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


#### With only one year of training data, there's no repetition for the model to learn a seasonal pattern from
This explains the slight changes in the model performances. We can experiment with this feature's implementation and the total data we use for training for more significant results.

analysis of it

## 8. Wildfire Risk Zone Feature

In [36]:
import requests
import geopandas as gpd

In [37]:
train_df, test_df = training_set.copy(), testing_set.copy()
train_df = merge_raw_lookup(train_df)
test_df = merge_raw_lookup(test_df)

In [38]:
def fetch_arcgis_layer(base_url, where="1=1", out_fields="*", page_size=1000):
    """Paginate through an ArcGIS REST layer's /query endpoint and return a GeoDataFrame."""
    features, offset = [], 0
    while True:
        params = {"where": where, "outFields": out_fields, "f": "geojson",
                   "resultOffset": offset, "resultRecordCount": page_size}
        resp = requests.get(base_url, params=params, timeout=60).json()
        batch = resp.get("features", [])
        if not batch:
            break
        features.extend(batch)
        offset += page_size
        print(f"  fetched {len(features)} features so far...")
        if len(batch) < page_size:
            break
    return gpd.GeoDataFrame.from_features(features)

In [39]:
FIRE_LAYER_URL = "https://services.gis.ca.gov/arcgis/rest/services/Environment/Fire_Severity_Zones/MapServer/0/query"

fire_zones_raw = fetch_arcgis_layer(FIRE_LAYER_URL)
print(f"Loaded {len(fire_zones_raw)} SRA fire hazard polygons")
print(fire_zones_raw.columns.tolist())

  fetched 1000 features so far...
  fetched 2000 features so far...
  fetched 3000 features so far...
  fetched 4000 features so far...
  fetched 5000 features so far...
  fetched 6000 features so far...
  fetched 7000 features so far...
  fetched 8000 features so far...
  fetched 9000 features so far...
  fetched 10000 features so far...
  fetched 11000 features so far...
  fetched 12000 features so far...
  fetched 13000 features so far...
  fetched 14000 features so far...
  fetched 15000 features so far...
  fetched 16000 features so far...
  fetched 17000 features so far...
  fetched 17267 features so far...
Loaded 17267 SRA fire hazard polygons
['geometry', 'OBJECTID', 'SRA', 'HAZ_CODE', 'HAZ_CLASS', 'Shape_Leng', 'Shape_Length', 'Shape_Area']


In [40]:
LRA_LAYER_URL = "https://services.gis.ca.gov/arcgis/rest/services/Environment/Fire_Severity_Zones/MapServer/1/query"

lra_zones_raw = fetch_arcgis_layer(LRA_LAYER_URL)
print(f"Loaded {len(lra_zones_raw)} LRA fire hazard polygons")
print(lra_zones_raw.columns.tolist())

  fetched 955 features so far...
Loaded 955 LRA fire hazard polygons
['geometry', 'OBJECTID_1', 'SRA', 'INCORP', 'HAZ_CODE', 'HAZ_CLASS', 'VH_REC', 'Shape_Leng', 'OBJECTID', 'FID_c19fhs', 'FHSZ_lyr', 'Shape_Length', 'Shape_Area']


In [41]:
FIRE_HAZARD_COL = "HAZ_CLASS"

fire_zones = fire_zones_raw.set_crs("EPSG:4326")[[FIRE_HAZARD_COL, "geometry"]]
lra_zones = lra_zones_raw.set_crs("EPSG:4326")[[FIRE_HAZARD_COL, "geometry"]]

def add_combined_fire_hazard(frame, sra_zones, lra_zones):
    points_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(frame["Longitude_raw"], frame["Latitude_raw"]),
        crs="EPSG:4326", index=frame.index,
    )

    # State Responsibility Area classification
    sra_joined = gpd.sjoin(points_gdf, sra_zones, how="left", predicate="within")
    sra_joined = sra_joined[~sra_joined.index.duplicated(keep="first")]
    result = pd.Series(np.nan, index=frame.index, dtype=object)
    result.loc[sra_joined.index] = sra_joined[FIRE_HAZARD_COL].values

    # for whatever SRA didn't cover, try Local Responsibility Area classification
    still_missing = result.isna()
    if still_missing.any():
        lra_joined = gpd.sjoin(points_gdf.loc[still_missing], lra_zones, how="left", predicate="within")
        lra_joined = lra_joined[~lra_joined.index.duplicated(keep="first")]
        result.loc[lra_joined.index] = lra_joined[FIRE_HAZARD_COL].values

    result = result.str.strip().str.title()

    print(f"Combined SRA+LRA match rate: {result.notna().mean():.1%}")
    return result.fillna("Not in Mapped Wildland/Urban Interface Zone") # If outside both

train_df["FireHazardZone"] = add_combined_fire_hazard(train_df, fire_zones, lra_zones)
test_df["FireHazardZone"] = add_combined_fire_hazard(test_df, fire_zones, lra_zones)

print(train_df["FireHazardZone"].value_counts())

Combined SRA+LRA match rate: 18.4%
Combined SRA+LRA match rate: 18.8%
FireHazardZone
Not in Mapped Wildland/Urban Interface Zone    66800
Very High                                      12286
Moderate                                        1389
High                                            1382
Name: count, dtype: int64


In [48]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw", "Latitude_raw", "Longitude_raw"]
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "HighSchoolDistrict", "Levels", "FireHazardZone"]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns]
# Tests only Fire Hazard Zone 

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "FireHazardZone")], ignore_index = True)
comparison_df

,model,train_r2,test_r2,test_mae,test_rmse,test_mape,New Features
0,linear regression,0.8534,0.8420,187946.0976,313321.9877,0.1904,Baseline
1,decision tree,0.8418,0.7961,195030.9801,355897.5189,0.1663,Baseline
2,random forest,0.9366,0.8734,143843.9497,280387.4237,0.1190,Baseline
3,linear regression,0.8547,0.8428,187109.9833,312499.9337,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,195612.6055,356542.6947,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,143588.6477,280461.0095,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,186061.2210,311142.1399,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,195510.3116,357268.4797,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,143345.3735,280532.5473,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,184492.5745,309355.5185,0.1835,BedBathRatio


# 9. Adding all features

In [54]:
def add_spatial_district(frame, districts):
    points_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(frame["Longitude_raw"], frame["Latitude_raw"]),
        crs="EPSG:4326",
        index=frame.index,
    )
    joined = gpd.sjoin(points_gdf, districts, how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]

    spatial_col = pd.Series(np.nan, index=frame.index, dtype=object)
    spatial_col.loc[joined.index] = joined["DistrictName"].values
    print(f"Real spatial match rate (before fallback): {spatial_col.notna().mean():.2%}")
    return spatial_col.fillna(frame["HighSchoolDistrict"])

train_df["SchoolDistrictSpatial"] = add_spatial_district(train_df,districts) # unedited Training and Testing Sets
test_df["SchoolDistrictSpatial"] = add_spatial_district(test_df, districts)

Real spatial match rate (before fallback): 99.89%
Real spatial match rate (before fallback): 99.95%


In [55]:
train_df["BedBathRatio"] = train_df["BedroomsTotal_raw"] / train_df["BathroomsTotalInteger_raw"].replace(0, np.nan)
test_df["BedBathRatio"] = test_df["BedroomsTotal_raw"] / test_df["BathroomsTotalInteger_raw"].replace(0, np.nan)

train_bed_bath_median = train_df["BedBathRatio"].median()
train_df["BedBathRatio"] = train_df["BedBathRatio"].fillna(train_bed_bath_median)
test_df["BedBathRatio"] = test_df["BedBathRatio"].fillna(train_bed_bath_median)

train_df["PropertyAgeAtSale"] = (train_df["CloseDate"].dt.year - train_df["YearBuilt_raw"]).clip(lower=0)
test_df["PropertyAgeAtSale"] = (test_df["CloseDate"].dt.year - test_df["YearBuilt_raw"]).clip(lower=0)

train_property_age_median = train_df["PropertyAgeAtSale"].median()
train_df["PropertyAgeAtSale"] = train_df["PropertyAgeAtSale"].fillna(train_property_age_median)
test_df["PropertyAgeAtSale"] = test_df["PropertyAgeAtSale"].fillna(train_property_age_median)

AMENITY_COLS = ["ViewYN", "PoolPrivateYN", "WaterfrontYN", "FireplaceYN", "NewConstructionYN"]
train_df["AmenityCount"] = train_df[AMENITY_COLS].sum(axis=1)
test_df["AmenityCount"] = test_df[AMENITY_COLS].sum(axis=1)

train_df["BuildingCoverageRatio"] = train_df["LivingArea"] / train_df["LotSizeSquareFeet"].replace(0, np.nan)
test_df["BuildingCoverageRatio"] = test_df["LivingArea"] / test_df["LotSizeSquareFeet"].replace(0, np.nan)
train_med_cov = train_df["BuildingCoverageRatio"].median()
train_df["BuildingCoverageRatio"] = train_df["BuildingCoverageRatio"].fillna(train_med_cov)
test_df["BuildingCoverageRatio"] = test_df["BuildingCoverageRatio"].fillna(train_med_cov)

In [56]:
COMP_K = 15     # nearest 15 neighbors by long and lat
coord_cols = ["Latitude_raw", "Longitude_raw"]
train_coords = train_df[coord_cols].values
train_prices = train_df["ClosePrice"].values

nn = NearestNeighbors(n_neighbors=COMP_K + 1, algorithm="ball_tree").fit(train_coords)
_, neighbor_idx = nn.kneighbors(train_coords)
neighbor_idx = neighbor_idx[:, 1:]  # drop self
train_df["CompPriceKNN"] = train_prices[neighbor_idx].mean(axis=1)

nn_test = NearestNeighbors(n_neighbors=COMP_K, algorithm="ball_tree").fit(train_coords)
_, test_neighbor_idx = nn_test.kneighbors(test_df[coord_cols].values)
test_df["CompPriceKNN"] = train_prices[test_neighbor_idx].mean(axis=1)

comp_scaler = StandardScaler()
train_df["CompPriceKNN"] = comp_scaler.fit_transform(train_df[["CompPriceKNN"]])
test_df["CompPriceKNN"] = comp_scaler.transform(test_df[["CompPriceKNN"]])

In [57]:
close_month_period = pd.PeriodIndex(train_df["CloseMonth"], freq="M")
test_month_period = pd.PeriodIndex(test_df["CloseMonth"], freq="M")

all_months_sorted = sorted(set(close_month_period) | set(test_month_period))
month_to_trend = {m: i for i, m in enumerate(all_months_sorted)}

train_df["MonthsSinceStart"] = close_month_period.map(month_to_trend)
test_df["MonthsSinceStart"] = test_month_period.map(month_to_trend)

train_df["MonthSin"] = np.sin(2 * np.pi * close_month_period.month / 12)
train_df["MonthCos"] = np.cos(2 * np.pi * close_month_period.month / 12)
test_df["MonthSin"] = np.sin(2 * np.pi * test_month_period.month / 12)
test_df["MonthCos"] = np.cos(2 * np.pi * test_month_period.month / 12)

In [58]:
NON_FEATURE_COLS = ["ListingKey", "CloseDate", "BedroomsTotal_raw", "BathroomsTotalInteger_raw", "YearBuilt_raw", "Latitude_raw", "Longitude_raw"]
CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor", "SpatialSchoolDistrict"] 
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in train_df.columns] 

X_train, y_train, feature_names = build_X_y(train_df,NON_FEATURE_COLS,CATEGORICAL_COLS)
X_test, y_test, _ = build_X_y(test_df,NON_FEATURE_COLS,CATEGORICAL_COLS)

comparison_df = pd.concat([comparison_df,evaluate_models(X_train, y_train, X_test, y_test, "Engineered Features")])

In [61]:
comparison_df 

,model,train_r2,test_r2,test_mape,New Features
0,linear regression,0.8534,0.8420,0.1904,Baseline
1,decision tree,0.8418,0.7961,0.1663,Baseline
2,random forest,0.9366,0.8734,0.1190,Baseline
3,linear regression,0.8547,0.8428,0.1891,SchoolDistrictSpatial
4,decision tree,0.8416,0.7954,0.1666,SchoolDistrictSpatial
5,random forest,0.9368,0.8734,0.1188,SchoolDistrictSpatial
6,linear regression,0.8535,0.8442,0.1876,SchoolDistrictSpatial (replacing HighSchoolDis...
7,decision tree,0.8414,0.7945,0.1664,SchoolDistrictSpatial (replacing HighSchoolDis...
8,random forest,0.9366,0.8733,0.1182,SchoolDistrictSpatial (replacing HighSchoolDis...
9,linear regression,0.8576,0.8459,0.1835,BedBathRatio


In [63]:
comparison_df.sort_values(by='test_r2', ascending=False).head(5)

,model,train_r2,test_r2,test_mape,New Features
23,random forest,0.9469,0.8850,0.1129,CompPriceKNN
35,random forest,0.9485,0.8841,0.1139,FireHazardZone
2,random forest,0.9483,0.8836,0.1140,Engineered Features
17,random forest,0.9369,0.8736,0.1190,AmenityCount
14,random forest,0.9367,0.8735,0.1189,PropertyAgeAtSale


# Work In Progress

- Potential flooding zones (same as above if applicable if data is available) **Failed**
- See if there are ratings, scorings, or potential sentiment analysis data for each school district used in the district geographic layer
- Experiment with variable creations (actually test with K is best for the CompPrice KNN and evaluate model performance directly after variable creation)